In [1]:
try:
    import google.colab  # noqa: F401

    # specify the version of DataEval (==X.XX.X) for versions other than the latest
    %pip install -q dataeval maite-datasets
except Exception:
    pass

In [2]:
import polars as pl
from maite_datasets.object_detection import SeaDrone

from dataeval import Metadata
from dataeval.config import set_max_processes
from dataeval.core import compute_stats
from dataeval.data import Limit, View
from dataeval.flags import ImageStats

# Statistics are calculated across a process pool. Capping it keeps this example's memory
# footprint predictable; leave it unset to use every available core.
set_max_processes(4)

In [3]:
# Load the SeaDrone dataset
sd_dataset = SeaDrone(root="./data", image_set="val", download=True)

# Limit to first 50 images for demonstration
dataset = View(sd_dataset, Limit(50))

print(f"Dataset size: {len(dataset)} images")
print(f"Sample image shape: {dataset[0][0].shape}")
print(f"Sample targets (boxes): {len(dataset[0][1].boxes)} boxes in first image")

Dataset size: 50 images
Sample image shape: (3, 933, 1230)
Sample targets (boxes): 1 boxes in first image


In [4]:
# Calculate custom individual statistics for full images only (per_image=True, per_target=False)
results_image_only = compute_stats(
    data=dataset,
    stats=ImageStats.PIXEL_MEAN | ImageStats.DIMENSION_ASPECT_RATIO | ImageStats.VISUAL_SHARPNESS,
    per_image=True,
    per_target=False,
    normalize_pixel_values=False,
)

print(f"Computed statistics: {list(results_image_only['stats'])}")
print(f"\nNumber of results: {len(results_image_only['source_index'])}")
print(f"Total images processed: {results_image_only['image_count']}")

Computed statistics: ['aspect_ratio', 'mean', 'sharpness']

Number of results: 50
Total images processed: 50


In [5]:
# Display first 5 source indices
print("First 5 SourceIndex entries (image-level only):")
for i, src in enumerate(results_image_only["source_index"][:5]):
    print(f"  {i}: item={src.item}, target={src.target}, channel={src.channel}")

print(f"\nAll entries have target=None: {all(src.target is None for src in results_image_only['source_index'])}")

First 5 SourceIndex entries (image-level only):
  0: item=0, target=None, channel=None
  1: item=1, target=None, channel=None
  2: item=2, target=None, channel=None
  3: item=3, target=None, channel=None
  4: item=4, target=None, channel=None

All entries have target=None: True


In [6]:
# Calculate basic pixel statistics for targets only (per_image=False, per_target=True)
results_target_only = compute_stats(
    data=dataset,
    stats=ImageStats.PIXEL_BASIC,
    per_image=False,
    per_target=True,
    normalize_pixel_values=False,
)

print(f"Computed statistics: {list(results_target_only['stats'])}")
print(f"Number of target-level results: {len(results_target_only['source_index'])}")
print(f"Total targets processed: {sum(results_target_only['object_count'])}")

# Display source indices for targets from first image
print("\nSourceIndex entries for targets in first few images:")
for i, src in enumerate(results_target_only["source_index"][:5]):
    print(f"  {i}: image={src.item}, target={src.target}, channel={src.channel}")

Computed statistics: ['mean', 'std', 'var']
Number of target-level results: 143
Total targets processed: 143

SourceIndex entries for targets in first few images:
  0: image=0, target=0, channel=None
  1: image=1, target=0, channel=None
  2: image=1, target=1, channel=None
  3: image=1, target=2, channel=None
  4: image=1, target=3, channel=None


In [7]:
# Calculate basic dimension statistics for full images and boxes (per_image=True, per_target=True)
results_both = compute_stats(
    data=dataset,
    stats=ImageStats.DIMENSION_BASIC,
    per_image=True,
    per_target=True,
    normalize_pixel_values=False,
)

print(f"Number of results (images + boxes): {len(results_both['source_index'])}")
print(f"Total images processed: {results_both['image_count']}")
print(f"Total boxes processed: {sum(results_both['object_count'])}")
print(f"Statistics calculated for each image: {list(results_both['stats'])}")

# Separate image-level and box-level results
image_indices = [i for i, src in enumerate(results_both["source_index"]) if src.target is None]
target_indices = [i for i, src in enumerate(results_both["source_index"]) if src.target is not None]

print(f"\nImage-level results: {len(image_indices)}")
print(f"Target-level results: {len(target_indices)}")

Number of results (images + boxes): 193
Total images processed: 50
Total boxes processed: 143
Statistics calculated for each image: ['width', 'height', 'channels', 'aspect_ratio']

Image-level results: 50
Target-level results: 143


In [8]:
results_background = compute_stats(
    data=dataset,
    stats=ImageStats.VISUAL_SHARPNESS,
    per_image=True,
    per_target=True,
    per_background=True,
    normalize_pixel_values=False,
)

print(f"Computed statistics: {sorted(results_background['stats'])}")

Computed statistics: ['background_fraction', 'background_sharpness', 'sharpness']


In [9]:
# The result spans image rows and target rows; only the image rows carry a background, so the
# target rows hold NaN here and have to be dropped before the fraction can be summarized.
fraction = pl.Series(results_background["stats"]["background_fraction"]).drop_nulls().drop_nans()

print(f"background_fraction: n={len(fraction)}  min={fraction.min():.3f}  median={fraction.median():.3f}")

background_fraction: n=50  min=0.968  median=0.998


In [10]:
metadata = Metadata(dataset)
supplied = set(metadata.factor_names)
metadata.add_factors(results_background)

print("levels      :", metadata.levels)
print("level counts:", metadata.level_counts)
print("new factors :", sorted(set(metadata.factor_names) - supplied))

levels      : ('unit', 'instance')
level counts: {'unit': 50, 'instance': 143}
new factors : ['instance_sharpness', 'unit_background_fraction', 'unit_background_sharpness', 'unit_sharpness']


/tmp/ipykernel_4829/2838730580.py:2: UserWarning: `date_time` was dropped: nearly every row holds a different value, so the column identifies rows rather than grouping them, and it is not numeric so there is no order along which to cut it into groups. Map the values onto a smaller vocabulary to keep the factor. See Metadata.dropped_factors.
  supplied = set(metadata.factor_names)


In [11]:
print("dropped:", metadata.dropped_factors)

print(
    metadata
    .rows_at("unit")
    .select("item_index", "unit_sharpness", "unit_background_sharpness", "unit_background_fraction")
    .head(5)
)

dropped: {'date_time': ['cardinality_over_budget'], 'instance_background_sharpness': ['no_values_at_level'], 'instance_background_fraction': ['no_values_at_level']}
shape: (5, 4)
┌────────────┬────────────────┬───────────────────────────┬──────────────────────────┐
│ item_index ┆ unit_sharpness ┆ unit_background_sharpness ┆ unit_background_fraction │
│ ---        ┆ ---            ┆ ---                       ┆ ---                      │
│ i64        ┆ f32            ┆ f32                       ┆ f32                      │
╞════════════╪════════════════╪═══════════════════════════╪══════════════════════════╡
│ 0          ┆ 25.101564      ┆ 25.021826                 ┆ 0.999166                 │
│ 1          ┆ 21.477745      ┆ 21.207449                 ┆ 0.997232                 │
│ 2          ┆ 19.58268       ┆ 19.114891                 ┆ 0.997434                 │
│ 3          ┆ 19.48291       ┆ 19.3801                   ┆ 0.999368                 │
│ 4          ┆ 21.473948      ┆ 21.414

/tmp/ipykernel_4829/2922025144.py:5: UserWarning: `altitude`, `frame`, `id`, `image_id`, `instance_sharpness`, `object_id`, `object_size`, `unit_background_fraction`, `unit_background_sharpness` and `unit_sharpness` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"altitude": [...]} to control this.
  .rows_at("unit")


In [12]:
units = metadata.rows_at("unit")
sharper = units.select((pl.col("unit_sharpness") > pl.col("unit_background_sharpness")).sum()).item()

print(f"images sharper as a whole than their own background: {sharper}/{len(units)}")
print(f"mean sharpness  object={metadata.rows_at('instance')['instance_sharpness'].mean():.1f}")
print(f"                image ={units['unit_sharpness'].mean():.1f}")
print(f"                back  ={units['unit_background_sharpness'].mean():.1f}")

images sharper as a whole than their own background: 48/50
mean sharpness  object=55.3
                image =22.5
                back  =22.0
